[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akifkoc/sabanci-qiskitfallfest26/blob/main/3-dolasiklik_ve_yogun_kodlama.ipynb)

# Bölüm 3: Dolaşıklık ve Yoğun Kodlama

**Qiskit ile Kuantum Programlamaya Giriş** · Qiskit Fall Fest 2026 · Sabancı Üniversitesi

Bu bölümün sonunda şunları yapabileceksiniz:

- H ve CNOT ile dolaşık (*entangled*) bir kubit çifti hazırlamak,
- Bell durumlarını tanımak,
- yoğun kodlama (*superdense coding*) ile tek kubit göndererek iki bitlik mesaj iletmek.

İki kubitlik durumları Qiskit sırasıyla, $|q_1 q_0\rangle$ biçiminde yazıyoruz. **Soru**ları önce kendiniz düşünün, sonra "Cevap"a tıklayın.

In [ ]:
# Google Colab'da bu hücreyi bir kez çalıştırın (yaklaşık 1 dakika sürer).
# Kendi bilgisayarınızda requirements.txt ile kurulum yaptıysanız bu hücreyi atlayabilirsiniz.
%pip install -q "qiskit[visualization]==2.6.0" "qiskit-aer==0.18.0"

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

simulator = AerSimulator()

## 1. Dolaşıklık

$q_0$'a H, sonra kontrolü $q_0$, hedefi $q_1$ olan bir CNOT uygulayalım:

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

display(Statevector(qc).draw("latex"))

qc.measure_all()
counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)
qc.draw("mpl")

**Soru:** İki kubitin ölçüm sonuçları arasında nasıl bir ilişki var?

<details>
<summary><b>Cevap</b></summary>

Ya `00` ya `11` görürüz: iki kubitin sonucu hep aynıdır. $q_0$'ı ölçüp $0$ görürsek $q_1$ de $|0\rangle$'a çöker; $1$ görürsek $|1\rangle$'e. Bu, kubitler birbirinden çok uzaktayken bile geçerlidir. Böyle kubitlere **dolaşık** (*entangled*) denir.

</details>

**Soru:** $\frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ durumu, her kubitin kendi durumunun tansör çarpımı $(a|0\rangle + b|1\rangle) \otimes (c|0\rangle + d|1\rangle)$ olarak yazılabilir mi?

<details>
<summary><b>Cevap</b></summary>

Hayır. Çarpım $ac|00\rangle + ad|01\rangle + bc|10\rangle + bd|11\rangle$'dir. $ad = bc = 0$ olmalı, ama o zaman $ac$ ya da $bd$ de sıfır olur. **Dolaşıklığın** tanımı budur: ortak durum, ayrı ayrı durumların tansör çarpımı olarak yazılamaz.

</details>

### Bell durumları

Bu durum dört **Bell durumundan** biridir:

$$ \frac{1}{\sqrt{2}}\big(|00\rangle + |11\rangle\big), \qquad \frac{1}{\sqrt{2}}\big(|00\rangle - |11\rangle\big), \qquad \frac{1}{\sqrt{2}}\big(|01\rangle + |10\rangle\big), \qquad \frac{1}{\sqrt{2}}\big(|01\rangle - |10\rangle\big) $$

Diğer üçü, ilkinden yalnızca $q_0$'a X ve/veya Z uygulanarak elde edilir.

### Görev 1

H ve CNOT'tan sonra $q_0$'a bir kapı ekleyerek $\frac{1}{\sqrt{2}}\big(|01\rangle + |10\rangle\big)$ durumunu hazırlayın.

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

# kodunuzu buraya yazın

Statevector(qc).draw("latex")

In [ ]:
# kontrol
target = (Statevector.from_label("01") + Statevector.from_label("10")) / np.sqrt(2)
if Statevector(qc).equiv(target):
    print("DOĞRU")
else:
    print("Henüz değil. İpucu: q0'ı çeviren kapıyı deneyin.")

## 2. Yoğun kodlama

Ediz ile Filiz dolaşık bir çift hazırlar: $q_0$ Ediz'de, $q_1$ Filiz'de kalır. Ediz, Filiz'e iki bitlik bir mesaj (`00`, `01`, `10` ya da `11`) göndermek istiyor, ama yalnızca **tek bir kubit** gönderecek.

**Ediz:** mesajın birinci karakteri `1` ise $q_0$'a X, ikinci karakteri `1` ise $q_0$'a Z uygular; sonra kubitini Filiz'e gönderir.

**Filiz:** CNOT (kontrol $q_0$, hedef $q_1$), sonra $q_0$'a H uygular ve iki kubiti ölçer. Sonuç, Ediz'in mesajıdır.

| Mesaj | Ediz | Ortak durum |
|---|---|---|
| `00` | — | $\frac{1}{\sqrt{2}}(\lvert 00\rangle + \lvert 11\rangle)$ |
| `01` | Z | $\frac{1}{\sqrt{2}}(\lvert 00\rangle - \lvert 11\rangle)$ |
| `10` | X | $\frac{1}{\sqrt{2}}(\lvert 01\rangle + \lvert 10\rangle)$ |
| `11` | X, sonra Z | $\frac{1}{\sqrt{2}}(\lvert 10\rangle - \lvert 01\rangle)$ |

(Son satır dördüncü Bell durumunun $-1$ katıdır; bütün durumu $-1$ ile çarpmak hiçbir ölçüm sonucunu değiştirmez.)

**Soru:** Filiz bu dört durumu nasıl ayırt edebilir?

<details>
<summary><b>Cevap</b></summary>

Dört Bell durumu birbirine **diktir**, bu yüzden kesin olarak ayırt edilebilir. Doğrudan ölçmek yetmez (ilk iki durum da `00` ya da `11` verir); Filiz'in CNOT ve H kapıları her durumu farklı bir baz durumuna götürür.

</details>

### Görev 2

Ediz `10` mesajını gönderiyor. Ediz'in ve Filiz'in kapılarını ekleyin. Filiz mesajı doğru okuyor mu? (Dikey kesikli çizgiler, `barrier`, yalnızca çizimi bölümlere ayırır.)

In [ ]:
qc = QuantumCircuit(2)

# dolaşık çift: q0 Ediz'de, q1 Filiz'de
qc.h(0)
qc.cx(0, 1)
qc.barrier()

# Ediz: kodunuzu buraya yazın (mesaj: 10)

qc.barrier()

# Filiz: kodunuzu buraya yazın (CNOT ve H)

qc.measure_all()
counts = simulator.run(qc, shots=1000).result().get_counts()
print("gönderilen: 10   Filiz'in okuduğu:", counts)
qc.draw("mpl")

In [ ]:
# kontrol
if counts == {"10": 1000}:
    print("DOĞRU")
else:
    print("Henüz değil. İpucu: Ediz q0'a X uygular; Filiz CNOT (kontrol q0, hedef q1) ve q0'a H uygular.")

Dört mesajın hepsini deneyelim:

In [ ]:
def superdense_coding(message):
    qc = QuantumCircuit(2)
    qc.h(0)              # dolaşık çift
    qc.cx(0, 1)
    if message[0] == "1":
        qc.x(0)          # Ediz
    if message[1] == "1":
        qc.z(0)          # Ediz
    qc.cx(0, 1)          # Filiz
    qc.h(0)              # Filiz
    qc.measure_all()
    return qc

for message in ["00", "01", "10", "11"]:
    counts = simulator.run(superdense_coding(message), shots=1000).result().get_counts()
    print("gönderilen:", message, "  Filiz'in okuduğu:", counts)

**Soru:** Ediz kaç kubit gönderdi, Filiz kaç bit okudu? Bunun bedeli ne?

<details>
<summary><b>Cevap</b></summary>

Ediz **bir** kubit gönderdi, Filiz **iki** bit okudu. Bedeli, önceden paylaşılmış dolaşık çifttir: Filiz'in kubiti de iletişimin parçasıdır.

</details>

## Özet

- H ve CNOT ile dolaşık bir çift hazırlanır: $\frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$.
- Dolaşık kubitlerin ölçüm sonuçları birbirine bağlıdır; ortak durum ayrı durumların tansör çarpımı olarak yazılamaz.
- Yoğun kodlamada Ediz, kendi kubitine X ve/veya Z uygulayarak dört Bell durumundan birini seçer; Filiz iki bitlik mesajı okur.

### Daha fazlası

- Qiskit: [ibm.com/quantum/qiskit](https://www.ibm.com/quantum/qiskit)
- IBM Quantum Learning, *Basics of Quantum Information*, "Superdense coding": [quantum.cloud.ibm.com/learning](https://quantum.cloud.ibm.com/learning/courses/basics-of-quantum-information/entanglement-in-action/superdense-coding)
- QWorld QBook101, "Entanglement and Superdense Coding": [qworld.net/qbook101](https://qworld.net/qbook101/)
- QTurkey (Kuantum Türkiye) etkinlikleri: [qturkey.org/events](https://qturkey.org/events/)